### Multi Token Prediction

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
class RMSNorm(nn.Module):
    def __init__(self, d_model:int, eps:float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))
    
    def forward(self,x):
        rms = torch.sqrt(x.pow(2).mean(dim=-1,keepdim = True) + self.eps)
        x_norm = x * (self.weight/rms)
        return x_norm  

In [15]:
class MultiTokenPrediction(nn.Module):
    def __init__(self, d_model: int, vocab_size: int, num_steps: int = 3, nhead: int = 2):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.num_steps = num_steps

        self.rms_norm = RMSNorm(d_model)

        self.embed = nn.Embedding(vocab_size, d_model)
        self.unembed = nn.Linear(d_model, vocab_size, bias=False)

        # weight tying
        self.unembed.weight = self.embed.weight

        self.projections = nn.ModuleList([
            nn.Linear(2 * d_model, d_model) for _ in range(num_steps)
        ])

        self.transformers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                batch_first=True
            )
            for _ in range(num_steps)
        ])

    def forward(self, token_ids: torch.LongTensor):
        b, t = token_ids.shape

        embeds = self.embed(token_ids)  # (b, t, d)

        outputs = []

        max_i = t - self.num_steps - 1

        for i in range(max_i + 1):
            h_prev = embeds[:, i, :]  # (b, d)

            logits_k = []

            for k in range(self.num_steps):
                future_pos = i + k + 1

                tok_embed = embeds[:, future_pos, :]  # (b, d)

                h_norm = self.rms_norm(h_prev)
                e_norm = self.rms_norm(tok_embed)

                merged = torch.cat([h_norm, e_norm], dim=-1)

                proj = self.projections[k](merged)  # (b, d)

                x = proj.unsqueeze(1)  # (b, 1, d)
                x = self.transformers[k](x)
                h_curr = x.squeeze(1)  # (b, d)

                logits = self.unembed(h_curr)  # (b, vocab)
                logits_k.append(logits)

                # iterative update (important design choice)
                h_prev = h_curr

            logits_k = torch.stack(logits_k, dim=1)  # (b, steps, vocab)
            outputs.append(logits_k)

        out = torch.stack(outputs, dim=1)  # (b, time, steps, vocab)

        return out
       

In [16]:
batch_size, seq_len, d_model, vocab_size = 1, 8, 8, 5000

model = MultiTokenPrediction(
    d_model=d_model,
    vocab_size=vocab_size,
    num_steps=3
)

tokens = torch.randint(0, vocab_size, (batch_size, seq_len))

logits = model(tokens)

print(f"Logits Shape: {logits.shape}")
# Expected: (batch, time, steps, vocab)

pred_ids = logits[0, 0].argmax(dim=-1)
print(f"Predicted tokens at position i=0 for all steps: {pred_ids}")

Logits Shape: torch.Size([1, 5, 3, 5000])
Predicted tokens at position i=0 for all steps: tensor([ 217, 3998, 3735])


In [17]:
batch_size , seq_len , vocab_size = 1,8,5000

targets = torch.randint(0,vocab_size, (batch_size,seq_len))
print(f"Targets Shape: {targets.shape}")

logits = model(tokens)
B,L,D,V = logits.shape
_ , T  = targets.shape

assert L == T-D
loss = 0.0

for i in range(L):
    for k in range(D):
        logit_ik = logits[:, i, k, :]            # (B, V)
        target_ik = targets[:, i + k + 1]        # (B,)
        loss += F.cross_entropy(logit_ik, target_ik)

loss = loss / (L * D)

print(f"Multi Token Prediction Loss: {loss.item()}")

Targets Shape: torch.Size([1, 8])
Multi Token Prediction Loss: 13.233903884887695
